In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, valid_ds, TRAIN_SIZE, VALID_SIZE, dice_metric, combined_loss, dice_ET, dice_TC, dice_WT

keras.mixed_precision.set_global_policy('mixed_float16')

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(1e-4)

inputs=keras.layers.Input(shape=(128,128,4))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_1_output=x

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_4_output=x

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(block_4_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

# 2x2 kernel upscaled from H->2H
x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_3_output]) # Up from 4->3
x=keras.layers.Dropout(0.2)(x)

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])
x=keras.layers.Dropout(0.2)(x)

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])
x=keras.layers.Dropout(0.2)(x)

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=16,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)

x=keras.layers.Conv2D(filters=16, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=16, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

outputs=keras.layers.Conv2D(4, kernel_size=(1,1), activation='softmax')(x)

model=keras.Model(inputs, outputs)

adam=keras.optimizers.Adam(1e-4, clipnorm=1.0)

sgd=keras.optimizers.SGD(1e-3,momentum=0.95, nesterov=True)

model.compile(
    optimizer=adam,
    loss=combined_loss,
    metrics=[dice_metric, dice_ET, dice_TC, dice_WT]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_metric',
    patience=3,
    factor=0.5,
    verbose=1,
    min_lr=5e-7,
    mode='max'
)

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=13,
    callbacks=[earlyStop_cb, lrPlateau_cb],
    validation_data=valid_ds,
    steps_per_epoch = 1500,
    validation_steps = 100
)

model.save('Saved Models/Seg_Adam_lr1e-4_LrPlateau_CombinedLoss_DiceMetric_DicePerClass.keras')

2026-04-19 18:06:31.226454: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-19 18:06:31.226651: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-19 18:06:31.226655: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-19 18:06:31.227209: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-19 18:06:31.227221: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/13


2026-04-19 18:06:34.224090: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-19 18:06:47.004991: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 209 of 256
2026-04-19 18:06:48.520625: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3709s 2s/step - dice_et: 0.3630 - dice_metric: 0.0746 - dice_tc: 0.0410 - dice_wt: 0.1044 - loss: 1.4437 - val_dice_et: 0.4995 - val_dice_metric: 0.0886 - val_dice_tc: 0.0485 - val_dice_wt: 0.1110 - val_loss: 1.2034 - learning_rate: 1.0000e-04
Epoch 2/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3129s 2s/step - dice_et: 0.4639 - dice_metric: 0.0894 - dice_tc: 0.0467 - dice_wt: 0.1099 - loss: 1.1504 - val_dice_et: 0.4997 - val_dice_metric: 0.0950 - val_dice_tc: 0.0528 - val_dice_wt: 0.1166 - val_loss: 1.1034 - learning_rate: 1.0000e-04
Epoch 3/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3108s 2s/step - dice_et: 0.5090 - dice_metric: 0.1168 - dice_tc: 0.0673 - dice_wt: 0.1434 - loss: 1.0651 - val_dice_et: 0.4650 - val_dice_metric: 0.2939 - val_dice_tc: 0.2631 - val_dice_wt: 0.3682 - val_loss: 1.0600 - learning_rate: 1.0000e-04
Epoch 4/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3131s 2s/step - dice_et: 0.5806 - dice_metric: 0.5333 - dice_tc: 0.5482 - dice_wt: 0.6150 - loss: 0.9619 - val_